# Baseline 1 — Structured Output من `Qwen2.5-Coder-14B-Instruct-GGUF`

## 1. Verify that GPU is Ready :

In [ ]:
!nvidia-smi


In [ ]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install huggingface_hub jsonschema


In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "Qwen/Qwen2.5-Coder-14B-Instruct-GGUF"
FILENAME = "qwen2.5-coder-14b-instruct-q5_k_m.gguf"

model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(" Now Model is ready:", model_path)


In [ ]:
import os
size_gb = os.path.getsize(model_path) / (1024**3)
print(f"File Size: {size_gb:.2f} GB")


## 4. Define JSON Schema related to Revit.

In [ ]:
import json

revit_command_schema = {
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "RevitCommand",
  "type": "object",
  "properties": {
    "action": {
      "type": "string",
      "enum": [
        "create_wall", "create_door", "create_window", "create_floor",
        "create_column", "create_level", "modify_parameter",
        "move_element", "delete_element", "query_elements", "unknown"
      ]
    },
    "target": {
      "type": "object",
      "properties": {
        "element_id": {"type": ["integer", "null"]},
        "category": {"type": ["string", "null"]},
        "filter": {"type": ["string", "null"]},
        "host_element_id": {"type": ["integer", "null"]}
      },
      "required": ["element_id", "category", "filter", "host_element_id"]
    },
    "parameters": {
      "type": "object",
      "properties": {
        "start_point": {"type": ["array", "null"], "items": {"type": "number"}, "minItems": 3, "maxItems": 3},
        "end_point": {"type": ["array", "null"], "items": {"type": "number"}, "minItems": 3, "maxItems": 3},
        "level": {"type": ["string", "null"]},
        "height": {"type": ["number", "null"]},
        "width": {"type": ["number", "null"]},
        "family_type": {"type": ["string", "null"]},
        "parameter_name": {"type": ["string", "null"]},
        "parameter_value": {"type": ["string", "null"]},
        "offset": {"type": ["array", "null"], "items": {"type": "number"}, "minItems": 3, "maxItems": 3},
        "elevation": {"type": ["number", "null"]}
      },
      "required": ["start_point", "end_point", "level", "height", "width",
                   "family_type", "parameter_name", "parameter_value", "offset", "elevation"]
    },
    "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
    "notes": {"type": ["string", "null"]}
  },
  "required": ["action", "target", "parameters", "confidence", "notes"]
}

print("schema v2 is ready (with host_element_id and elevation)")


In [ ]:
from llama_cpp import Llama, LlamaGrammar

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False,
)

grammar = LlamaGrammar.from_json_schema(json.dumps(revit_command_schema))
print("model and grammer are ready ].")


## 6. تعريف الـ System Prompt (بدون few-shot الأول، عشان نتأكد إن الموديل نفسه سليم)

In [ ]:
SYSTEM_PROMPT = (
    "You are a converter that turns natural-language Revit instructions "
    "into a single structured JSON command. Only output valid JSON matching "
    "the given schema. If the instruction is ambiguous, make a reasonable "
    "assumption and explain it in \"notes\". If the instruction does not "
    "correspond to any supported action, use action = \"unknown\".\n\n"
    "Field usage rules:\n"
    "- For create_door / create_window: put the host wall's id in "
    "target.host_element_id, NOT target.element_id (element_id is only for "
    "modify/move/delete/query actions that target an existing element directly).\n"
    "- For create_level: use parameters.elevation for the level's vertical "
    "position. NEVER use parameters.height for this - height is only for the "
    "height dimension of an object like a wall or door.\n"
    "- For modify_parameter: ALWAYS use parameters.parameter_name and "
    "parameters.parameter_value together to describe the change, even if the "
    "parameter being changed has a typed field elsewhere in the schema (like "
    "height or width). Leave those typed fields null in this case - do not "
    "fill both the typed field and parameter_name/parameter_value."
)
print("تم تعريف SYSTEM_PROMPT (v2: قواعد host_element_id / elevation / modify_parameter)")


## 7. فحص عقلانية سريع (من غير grammar)

قبل ما نستثمر وقت في الـ batch، نتأكد إن الموديل ده أصلاً بيرد بمنطق.

In [ ]:
raw = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Create a wall from point (0,0,0) to (5000,0,0) on Level 1 with height 3000mm"},
    ],
    max_tokens=400,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    repeat_penalty=1.1,
)
print(raw["choices"][0]["message"]["content"])


## 8. Few-shot prompting + grammar (النسخة اللي هنستخدمها فعليًا)

بما إن الموديل ده عمومي (مش متدرب على Revit تحديدًا)، الأمثلة دي بتوريه شكل الـ mapping
المطلوب بين التعليمة والـ JSON.

In [ ]:
FEWSHOT_SYSTEM_PROMPT = SYSTEM_PROMPT + """

Examples:

Instruction: "Create a wall from (0,0,0) to (3000,0,0) on Level 1 with height 2800mm"
Output: {"action": "create_wall", "target": {"element_id": null, "category": null, "filter": null, "host_element_id": null}, "parameters": {"start_point": [0,0,0], "end_point": [3000,0,0], "level": "Level 1", "height": 2800, "width": null, "family_type": null, "parameter_name": null, "parameter_value": null, "offset": null, "elevation": null}, "confidence": "high", "notes": null}

Instruction: "Add a door type 'Single-Flush: 900x2100mm' on the wall with id 45021"
Output: {"action": "create_door", "target": {"element_id": null, "category": null, "filter": null, "host_element_id": 45021}, "parameters": {"start_point": null, "end_point": null, "level": null, "height": null, "width": null, "family_type": "Single-Flush: 900x2100mm", "parameter_name": null, "parameter_value": null, "offset": null, "elevation": null}, "confidence": "high", "notes": null}

Instruction: "Create a new level called 'Level 4' at elevation 12000mm"
Output: {"action": "create_level", "target": {"element_id": null, "category": null, "filter": null, "host_element_id": null}, "parameters": {"start_point": null, "end_point": null, "level": "Level 4", "height": null, "width": null, "family_type": null, "parameter_name": null, "parameter_value": null, "offset": null, "elevation": 12000}, "confidence": "high", "notes": null}

Instruction: "Change the height parameter of wall id 45010 to 4000mm"
Output: {"action": "modify_parameter", "target": {"element_id": 45010, "category": null, "filter": null, "host_element_id": null}, "parameters": {"start_point": null, "end_point": null, "level": null, "height": null, "width": null, "family_type": null, "parameter_name": "Height", "parameter_value": "4000", "offset": null, "elevation": null}, "confidence": "high", "notes": null}

Instruction: "Show me all doors on Level 2"
Output: {"action": "query_elements", "target": {"element_id": null, "category": "Doors", "filter": "level = Level 2", "host_element_id": null}, "parameters": {"start_point": null, "end_point": null, "level": null, "height": null, "width": null, "family_type": null, "parameter_name": null, "parameter_value": null, "offset": null, "elevation": null}, "confidence": "high", "notes": null}
"""

def generate_structured_output(instruction: str, system_prompt: str = FEWSHOT_SYSTEM_PROMPT) -> dict:
    out = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": instruction},
        ],
        grammar=grammar,
        max_tokens=512,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repeat_penalty=1.1,
    )
    text = out["choices"][0]["message"]["content"].strip()
    return json.loads(text, strict=False)

result = generate_structured_output(
    "Create a wall from point (0,0,0) to (5000,0,0) on Level 1 with height 3000mm"
)
print(json.dumps(result, indent=2, ensure_ascii=False))


## 9. اختبار دفعة من الأوامر + قياس نسبة نجاح الـ schema validity

In [ ]:
from jsonschema import validate, ValidationError
import time

sample_instructions = [
    "Create a wall from point (0,0,0) to (5000,0,0) on Level 1 with height 3000mm",
    "Add a door with type \'Single-Flush: 900x2100mm\' at element id 45021",
    "Create a window on the wall with element id 45010, family type \'Fixed: 1200x1200mm\'",
    "Move the column with id 39012 by 500mm in the x direction",
    "Delete all elements in category Furniture on Level 2",
    "Change the height parameter of wall id 45010 to 4000mm",
    "Create a floor on Level 1 with family type \'Generic 300mm\'",
    "Show me all doors on Level 3",
    "Create a new level called \'Level 4\' at elevation 12000mm",
    "Set the comments parameter of element 51022 to \'Needs review\'",
]

results = []
valid_count = 0

for i, instruction in enumerate(sample_instructions, 1):
    start = time.time()
    try:
        output = generate_structured_output(instruction)
        validate(instance=output, schema=revit_command_schema)
        valid = True
        valid_count += 1
    except (ValidationError, json.JSONDecodeError) as e:
        output = str(e)
        valid = False
    elapsed = round(time.time() - start, 2)
    print(f"[{i}/{len(sample_instructions)}] valid={valid} ({elapsed}s) -> {instruction}")
    results.append({
        "instruction": instruction,
        "output": output,
        "schema_valid": valid,
        "seconds": elapsed,
    })

rate = round(100 * valid_count / len(sample_instructions), 1)
print(f"\nSuccess with (schema validity): {valid_count}/{len(sample_instructions)} = {rate}%")


## 10. Save Results 

In [ ]:
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Saved in results.json")
